In [1]:
#!pip -q install -U vllm
!pip uninstall -y vllm
!pip install -q --no-cache-dir "vllm==0.19.1"
!pip -q install -U transformers huggingface_hub
# A100 에서 돌릴때만 실행
!pip install -U "protobuf>=5.26.1,<6"
#!pip show vllm

Found existing installation: vllm 0.19.1
Uninstalling vllm-0.19.1:
  Successfully uninstalled vllm-0.19.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 272.1 MB/s eta 0:00:00


In [1]:
import torch, os
print(torch.__version__)
print(torch.version.cuda)
print(os.popen("nvidia-smi").read())

2.10.0+cu128
12.8
Mon Jun  1 07:34:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   32C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------

In [2]:
!pip show vllm

Name: vllm
Version: 0.19.1
Summary: A high-throughput and memory-efficient inference and serving engine for LLMs
Home-page: https://github.com/vllm-project/vllm
Author: vLLM Team
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, anthropic, blake3, cachetools, cbor2, cloudpickle, compressed-tensors, depyf, diskcache, einops, fastapi, filelock, flashinfer-cubin, flashinfer-python, gguf, ijson, lark, llguidance, lm-format-enforcer, mcp, mistral_common, model-hosting-container-standards, msgspec, ninja, numba, numpy, nvidia-cudnn-frontend, nvidia-cutlass-dsl, openai, openai-harmony, opencv-python-headless, opentelemetry-api, opentelemetry-exporter-otlp, opentelemetry-sdk, opentelemetry-semantic-conventions-ai, outlines_core, partial-json-parser, pillow, prometheus-fastapi-instrumentator, prometheus_client, protobuf, psutil, py-cpuinfo, pybase64, pydantic, python-json-logger, pyyaml, pyzmq, quack-kernels, regex, requests, sentencepiece, setproctit

In [3]:

# 모델로딩

import os
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams

model_id = 'Qwen/Qwen3.5-35B-A3B-GPTQ-Int4'

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    tensor_parallel_size=1,
)

from transformers import AutoTokenizer

qwen_tok = AutoTokenizer.from_pretrained(
    model_id,
    trust_remote_code=True,
)


INFO 06-01 08:40:49 [utils.py:233] non-default args: {'trust_remote_code': True, 'disable_log_stats': True, 'model': 'Qwen/Qwen3.5-35B-A3B-GPTQ-Int4'}
WARNING 06-01 08:40:49 [envs.py:1744] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 06-01 08:40:52 [model.py:549] Resolved architecture: Qwen3_5MoeForConditionalGeneration
INFO 06-01 08:40:52 [model.py:1678] Using max model len 262144
INFO 06-01 08:40:54 [gptq_marlin.py:229] The model is convertible to gptq_marlin during runtime. Using gptq_marlin kernel.
INFO 06-01 08:40:54 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.


[transformers] `Qwen2VLImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Qwen2VLImageProcessor` instead.


INFO 06-01 08:40:54 [config.py:281] Setting attention block size to 1056 tokens to ensure that attention page size is >= mamba page size.
INFO 06-01 08:40:54 [config.py:312] Padding mamba page size by 0.76% to ensure that mamba page size and attention page size are exactly equal.


Parse safetensors files:   0%|          | 0/14 [00:00<?, ?it/s]

INFO 06-01 08:40:57 [vllm.py:790] Asynchronous scheduling is enabled.


[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.


In [6]:
'''데이터 로딩'''

import pandas as pd
import json

tmp = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/kisti/data/tmp2.csv')

In [7]:
''' 프롬프트 구성 '''

import ast
import json
import pandas as pd


def parse_struct(x):

    if x is None:
        return None

    # pandas NaN 처리
    if isinstance(x, float) and pd.isna(x):
        return None

    if isinstance(x, (list, dict)):
        return x

    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None

        # JSON 문자열 시도
        try:
            return json.loads(x)
        except Exception:
            pass

        # Python literal 문자열 시도
        try:
            return ast.literal_eval(x)
        except Exception:
            return x

    return x

def extract_group(x):
    x = parse_struct(x)

    if not isinstance(x, dict):
        return None

    group = x.get("group")

    if group is None:
        return None

    s = str(group).strip()
    if not s or s.lower() in {"nan", "none", "null"}:
        return None

    return s

def normalize_str_list(x):
    """
    문자열/리스트/문자열화된 리스트를
    ['a', 'b', ...] 형태로 정규화
    """
    x = parse_struct(x)

    if x is None:
        return []

    if not isinstance(x, list):
        x = [x]

    result = []
    for v in x:
        if v is None:
            continue
        s = str(v).strip()
        if s:
            result.append(s)

    return result



SYSTEM_PROMPT = (
    "너는 설명을 생성하는 한국어 AI야. "
    "입력으로 제공된 JSON 데이터만 사용해. "
    "단, 입력에 있는 키워드나 목적 용어는 의미를 바꾸지 않는 범위에서 일반인이 이해하기 쉬운 말로 풀어 써. "
    "다음 규칙을 따라 최종 결과만 JSON으로 출력하라."
    "중간 과정은 출력하지 마라."
    "1) 과제 정보인 경우 다음 절차를 따른다:\n"
    "   1-1) project.title, project.keyword에서 과제의 핵심 대상과 기술 방향을 추출한다.\n"
    "        - project.keyword 값이 없으면 해당 내용은 완전히 생략하고 값이 없다는 설명은 하지 않는다.\n"
    "   1-2) related_research.paper 또는 related_research.patent에 값이 없으면 논문 및 특허는 언급하지 않는다.\n"
    "        - 값이 없다는 사실이나 부족하다는 점을 설명하는 문장은 절대 작성하지 않는다.\n"
    "   1-3) 위 정보를 종합해 다음을 정리한다:\n"
    "        - 과제의 목적 또는 개발 대상\n"
    "        - 과제와 관련된 주요 기술 분야 또는 방법\n"
    "        - 논문 또는 특허가 있다면 이를 통해 볼 때 집중하는 핵심 기술 또는 연구 방향\n"
    "   1-4) project.project_info에 값이 있는 경우 값이 있는 해당 정보만 다음과 같이 반드시 포함한다:\n"
    "        - performing_org: 해당 기관이 수행하는 과제임을 자연스럽게 포함한다.\n"
    "        - science_category: 해당 과제가 어떤 기술 분야에 속하는지 자연스럽게 포함한다.\n"
    "        - total_research_fund_top_ratio와 total_research_fund_group 값이 있으면, total_research_fund_group이 '전체'인 경우 '총연구비는 전체의 상위 ○% 수준', 그 외에는 '총연구비는 {group} 업종 내 상위 ○% 수준'처럼 자연스럽게 포함한다.\n"
    "        - 비율 정보는 group 값과 함께 있을 때만 사용해 '전체의 상위 ○%' 또는 '{group} 업종 내 상위 ○%' 형태로 표현하며, 둘 중 하나라도 없으면 해당 정보는 완전히 생략한다.\n"
    "        - total_research_period: 값이 있으면 입력된 값을 그대로 활용해 총연구기간을 자연스럽게 포함한다.\n"
    "        - 위 정보는 값이 있을 때만 언급하고, 값이 없으면 완전히 생략한다.\n"
    "2) 문장 작성 규칙:\n"
    "   - 기술 용어는 일반인이 이해할 수 있도록 쉬운 표현으로 풀어 쓴다.\n"
    "   - 각 섹션은 output_requirements.section_sentence_range 범위를 지켜 작성한다.\n"
    "3) 금지 규칙:\n"
    "   - 추측, 확장, 전망, 예상, 가능성, 효과 단정 등 입력에 없는 내용은 작성하지 않는다.\n"
    "   - 입력값이 없다는 사실을 설명하거나 부족함을 언급하는 문장을 작성하지 않는다.\n"
    "   - '정보가 없다', '데이터가 없다', '제공되지 않았다', '확인할 수 없다', '설명하기 어렵다'와 같은 표현은 출력하지 않는다.\n"
    "   - output_requirements.forbidden에 포함된 단어는 출력 어디에도 사용하지 않는다.\n"
    "4) 내부 사고 과정은 절대 출력하지 않고, 최종 결과만 출력한다.\n"
    "5) 과제 설명에는 입력된 project.title와 performing_org 값을 그대로 사용한다.\n"
    "   - 과제 설명에서는 입력에 없는 과제명 및 수행기관명으로 바꾸지 않는다.\n"
    "6) 입력값이 없거나 비어 있는 항목(None, NaN, 빈 문자열, 빈 리스트)은 없는 정보로 간주하고 완전히 무시한다.\n"
    "   - 없는 정보에 대해서는 언급 자체를 하지 않는다.\n"
)

FEWSHOT_MESSAGES = [
    {
        "role": "user",
        "content": (
            "아래 JSON을 기반으로 과제 설명을 작성해.\n"
            "출력은 JSON 하나만 반환해. JSON 외 텍스트 금지.\n"
            "키는 output_requirements.format에 있는 섹션 제목을 그대로 사용해.\n\n"
            + json.dumps({
                "project": {
                    "project_id": "P001",
                    "title": "액정 엘라스토머 기반 4D 프린팅 소재 개발",
                    "keyword": [
                        "코팅공정용", "용액공정", "금속입자소결체", "잉크토출장치", "공정기술"
                    ],
                    "project_info": {
                        "performing_org": "테스트기관베타7X",
                        "science_category": "주조/용접/접합",
                        "total_research_fund_top_ratio": "5",
                        "total_research_fund_group": "전체",
                        "total_research_period": "8개월 30일"
                    }
                },
                "related_research": {
                    "paper": ['습윤 고분자 탄성 액추에이터의 4D 프린팅'],
                    "patent": ['폴리로탁산 가교체를 도입한 액정 엘라스토머 필름의 제조 방법']
                },
                "output_requirements": {
                    "language": "ko",
                    "format": [
                        "과제 설명"
                    ],
                    "section_sentence_range": "각 섹션 4~7문장",
                    "forbidden": ["예시", "참고", "제출", "메타"]
                }
            }, ensure_ascii=False)
        )
    },
    {
        "role": "assistant",
        "content": json.dumps({
            "과제 설명": (
                "이 과제는 테스트기관베타7X에서 수행했으며, 주조/용접/접합 분야에 속합니다."
                "또한, 총연구비는 전체의 상위 5%에 속하며, 총연구기간은 8개월 30일입니다."
                "이 과제는 액정 엘라스토머라는 소재를 활용해 시간이나 자극에 따라 형태가 변하는 4D 프린팅용 재료를 개발하는 것을 목표로 합니다."
                "이를 위해 액체 상태의 재료를 원하는 위치에 정밀하게 뿌리거나 코팅하는 공정 기술이 활용됩니다."
                "특히 금속 입자를 포함한 재료와 잉크 분사 장치, 그리고 전체 공정을 제어하는 기술을 통해 소재를 형성하는 방식과 연결됩니다."
                "또한 관련 연구에서는 형태 변화가 가능한 고분자 기반 소재와 필름 구조를 만드는 방법이 다루어져, 이러한 소재 구조를 구현하는 데 초점이 맞춰져 있습니다."
                "전체적으로 이 과제는 정밀한 용액 공정과 인쇄 기술을 기반으로 형태 변화가 가능한 기능성 소재를 만드는 데 목적이 있습니다."
            ),
        }, ensure_ascii=False)
    }
]


def build_project_prompt(rec_row):

    pid = rec_row["project_id"]
    pname = rec_row["project_name"]
    keyword_proj = normalize_str_list(rec_row.get("keyword_project", []))

    paper_list = normalize_str_list(rec_row.get("paper", []))
    patent_list = normalize_str_list(rec_row.get("patent", []))

    project_info = {
        "performing_org": rec_row.get("과제수행기관명"),
        "science_category": rec_row.get("과학기술표준분류1-중"),
        "total_research_fund_top_ratio": rec_row.get("총연구비_상위비율"),
        "total_research_fund_group": extract_group(rec_row.get("총연구비_판정정보")),
        "total_research_period": rec_row.get("총연구기간"),
    }

    fmt = ["과제 설명"]

    payload = {
        "project": {
            "project_id": pid,
            "title": pname,
            "keyword": keyword_proj,
            "project_info": project_info
        },
        "output_requirements": {
            "language": "ko",
            "format": fmt,
            "section_sentence_range": "섹션 4~7문장",
            "forbidden": ["예시", "참고", "제출", "메타"]
        }
    }

    related = {}
    if paper_list:
        related["paper"] = paper_list
    if patent_list:
        related["patent"] = patent_list

    if related:
        payload["related_research"] = related

    user_prompt = (
        "JSON을 기반으로 과제 설명을 작성해.\n"
        "출력은 JSON 하나만 반환해. JSON 외 텍스트 금지야.\n"
        "few-shot 예시에 나온 기관명, 과제명을 그대로 쓰지 말고 현재 입력 JSON의 값만 사용해.\n"
        "few-shot 예시에 나온 모든 고유명사는 예시 전용이며, 현재 출력에 절대 재사용하지 마라.\n"
        "첫 글자는 {, 마지막 글자는 } 로 끝내.\n"
        "``` 같은 코드블록은 절대 쓰지 마.\n"
        "입력에서 값이 없거나 비어 있는 항목은 완전히 생략하고, 값이 없다는 설명은 절대 쓰지 마.\n"
        "키는 output_requirements.format에 있는 섹션 제목을 그대로 사용해.\n"
        f"{json.dumps(payload, ensure_ascii=False, default=str)}"
    )

    return (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + FEWSHOT_MESSAGES
        + [{"role": "user", "content": user_prompt}]
    )



In [8]:
'''LLM 모델 호출 후 설명 생성 함수'''

import torch, gc
import re
from vllm import SamplingParams


def _messages_to_prompt(messages, tokenizer):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


@torch.inference_mode()
def generate_explanation(messages, tokenizer, model,
                         max_new_tokens=1024):

    prompt = _messages_to_prompt(messages, tokenizer)

    json_schema = {
        "type": "object",
        "properties": {
            "과제 설명": {
                "type": "string"
            }
        },
        "required": ["과제 설명"],
        "additionalProperties": False
    }

    structured_outputs = StructuredOutputsParams(
        json=json_schema
    )

    params = SamplingParams(
        temperature=0.0,
        top_p=1.0,
        max_tokens=max_new_tokens,
        stop=["<|im_end|>"],
        structured_outputs=structured_outputs
    )

    outputs = model.generate([prompt], params)
    raw = outputs[0].outputs[0].text.strip()

    return raw, raw, 1

<>:47: SyntaxWarning: invalid escape sequence '\s'
<>:47: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_30243/2076349077.py:47: SyntaxWarning: invalid escape sequence '\s'
  result = re.sub(r"<think>.*?</think>\s*", "", result, flags=re.DOTALL).strip()


In [9]:
# '''설명 생성 실행''' (MoE 모델)

import json
from itertools import islice
import re
import os
import time
import pandas as pd

# 입력 토큰 수 확인
def count_prompt_tokens(messages, tok):
    '''
    prompt = _messages_to_prompt(messages)
    # add_special_tokens=False: 이미 <|im_start|> 같은 토큰을 직접 넣고 있어서 중복 방지
    ids = tok(prompt, add_special_tokens=False).input_ids
    return len(ids)
    '''
    prompt = _messages_to_prompt(messages, tok)

    ids = tok(
        prompt,
        add_special_tokens=False
    ).input_ids

    return len(ids)

out_path = '/content/drive/MyDrive/Colab Notebooks/kisti/result/exp_project.csv'

# 기존 파일 삭제
if os.path.exists(out_path):
    os.remove(out_path)
file_exists = os.path.exists(out_path)  # False

start_time = time.time()

project_times = []

for project_id, block in tmp.groupby("project_id", sort=False): #(project_id = 과제고유번호)
    project_time_start = time.time()

    project_name = block["project_name"].iloc[0] #(project_name = 과제명)
    project_row = block.iloc[0]
    explanations = []
    project_rows = []

    project_desc_cached = None

    messages = build_project_prompt(project_row)

    prompt_tokens = count_prompt_tokens(messages, qwen_tok)
    if prompt_tokens > 30000:
        continue

    print("prompt_tokens =", prompt_tokens)

    #one = generate_explanation(messages, tokenizer=None, model=None)
    one, raw, think_closed = generate_explanation(
        messages,
        tokenizer=qwen_tok,
        model=llm
    )

    generated_tokens = len(
        #qwen_tok(one, add_special_tokens=False).input_ids
        qwen_tok(raw, add_special_tokens=False).input_ids
    )

    print(
        f"generated_tokens={generated_tokens}"
    )

    valid_json = 1
    parsed_json = ""
    project_desc = ""

    try:
        obj = json.loads(one)
        parsed_json = json.dumps(obj, ensure_ascii=False)

        if isinstance(obj, dict):
            project_desc = obj.get("과제 설명", "")

    except Exception:
        valid_json = 0

    project_time = time.time() - project_time_start

    print(
        f"===========project_name: {project_name} // "
        f"project_id: {project_id} // "
        f"time: {time.strftime('%H:%M:%S', time.gmtime(project_time))}=========="
    )

    project_times.append(project_time)

    project_rows = [{
        "project_name": project_name,
        "project_id": project_id,
        "project_description": project_desc,
        "raw_output": raw,
        "clean_output": one,
        "think_closed": think_closed,
        "valid_json": valid_json
    }]

    explain_df = pd.DataFrame(project_rows)

    explain_df.to_csv(
        out_path,
        mode="a",
        header=not file_exists,
        index=False,
        encoding="utf-8-sig"
    )

    file_exists = True

prompt_tokens = 1615


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

generated_tokens=216
===========project_name: 다양한 형태 무게 강도의 불특정 물체를 견고하게 파지 가능한 다품종 생산 공정용 그리퍼 시스템 개발 // project_id: 1415180849 // time: 00:00:03==========
prompt_tokens = 1748


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

generated_tokens=253
===========project_name: 통합형 최적설계 플랫폼 기반 초고효율 전력변환시스템 개발 // project_id: 1415180044 // time: 00:00:01==========
prompt_tokens = 1627


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

generated_tokens=200
===========project_name: 분기부 Anti-Icing 시스템 개발 // project_id: 1615012274 // time: 00:00:01==========
